# Qwen semantic causal hallucination diagnosis

This notebook adds a semantic readout to the existing Qwen2.5-VL causal heatmaps.

It does **not** rerun all patch perturbations. It reuses the causal patch rankings already computed for Qwen 7B and 3B, then asks:

1. What object concept is represented at the top causal visual patch?
2. Is the queried object represented more strongly at the causal patch than at a matched random patch?
3. For hallucinations, does the semantic readout explain the false visual cue?
4. Does a semantic layer chosen on Qwen 7B validation transfer to Qwen 3B by relative depth?

Required files in `/workspace`:

- `pope_confidence_category_matched_RESULTS_20260718_062034.zip`
- `pope_crossmodel_3b_sufficiency_confirmation_RESULTS.zip`

Expected output:

`/workspace/qwen_semantic_causal_hallucination_diagnosis_RESULTS.zip`


In [ ]:
# Cell 1: install
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "transformers==4.50.3", "accelerate", "datasets==2.19.2",
    "huggingface_hub", "safetensors", "sentencepiece",
    "numpy==1.26.4", "pandas==2.2.2", "scipy==1.11.4",
    "scikit-learn==1.4.2", "pillow", "matplotlib", "tqdm"
])
print("Restart the kernel, then run Cell 2.")


In [ ]:
# Cell 2: setup, extract prior results, and prepare examples

import gc, json, math, os, random, re, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset

WORKSPACE = Path("/workspace")
PROJECT = WORKSPACE / "qwen_semantic_causal_hallucination_diagnosis"
INPUT = PROJECT / "input"
OUTPUT = PROJECT / "outputs"
CACHE = PROJECT / "cache"
for p in [PROJECT, INPUT, OUTPUT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def find_zip(patterns):
    matches = []
    for pattern in patterns:
        matches.extend(WORKSPACE.glob(pattern))
    matches = [p for p in matches if p.is_file() and zipfile.is_zipfile(p)]
    return max(matches, key=lambda p: p.stat().st_size) if matches else None

def extract_once(path, dest):
    marker = dest / ".complete"
    if marker.exists():
        return
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as z:
        z.extractall(dest)
    marker.write_text(path.name)

z7 = find_zip([
    "pope_confidence_category_matched_RESULTS_20260718_062034.zip",
    "*confidence*category*matched*RESULTS*.zip",
])
z3 = find_zip([
    "pope_crossmodel_3b_sufficiency_confirmation_RESULTS.zip",
    "*crossmodel*3b*sufficiency*RESULTS*.zip",
])
if z7 is None or z3 is None:
    raise FileNotFoundError("Upload both required result ZIPs to /workspace.")

d7, d3 = INPUT / "qwen7b", INPUT / "qwen3b"
extract_once(z7, d7)
extract_once(z3, d3)

sel7 = pd.read_csv(d7 / "outputs" / "selected_confidence_category_matched.csv")
dist7 = pd.read_csv(d7 / "outputs" / "grid_patch_layer_distances.csv")
patch7 = pd.read_csv(d7 / "outputs" / "grid_patch_results.csv")
sel3 = pd.read_csv(d3 / "selected_crossmodel_confirmation.csv")
patch3 = pd.read_csv(d3 / "crossmodel_heatmap_results.csv")

# Use all available validation and evaluation examples.
LAYER_7B = 21
top7 = (
    dist7[dist7.layer.astype(int) == LAYER_7B]
    .sort_values(["selected_index", "relative_l2"], ascending=[True, False])
    .groupby("selected_index", as_index=False)
    .first()[["selected_index", "patch_index", "relative_l2"]]
    .rename(columns={"patch_index":"top_patch", "relative_l2":"top_patch_score"})
)
examples7 = sel7.merge(top7, on="selected_index", how="inner")
examples7["model_name"] = "qwen7b"
examples7["model_id"] = "Qwen/Qwen2.5-VL-7B-Instruct"
examples7["causal_layer"] = LAYER_7B

top3 = (
    patch3.sort_values(["selected_index", "relative_l2"], ascending=[True, False])
    .groupby("selected_index", as_index=False)
    .first()[["selected_index", "patch_index", "relative_l2"]]
    .rename(columns={"patch_index":"top_patch", "relative_l2":"top_patch_score"})
)
examples3 = sel3.merge(top3, on="selected_index", how="inner")
examples3["model_name"] = "qwen3b"
examples3["model_id"] = "Qwen/Qwen2.5-VL-3B-Instruct"
examples3["causal_layer"] = 27

common = [
    "model_name","model_id","selected_index","row_idx","question",
    "target_category","clean_group","split","top_patch","top_patch_score",
    "causal_layer"
]
for frame in [examples7, examples3]:
    for col in common:
        if col not in frame.columns:
            frame[col] = np.nan

examples = pd.concat([examples7[common], examples3[common]], ignore_index=True)
examples["row_idx"] = pd.to_numeric(examples.row_idx).astype(int)
examples["selected_index"] = pd.to_numeric(examples.selected_index).astype(int)
examples["top_patch"] = pd.to_numeric(examples.top_patch).astype(int)
examples["is_grounded"] = (examples.clean_group.astype(str) == "grounded_yes").astype(int)

def random_patch(row):
    rng = np.random.default_rng(SEED + int(row.row_idx)*101 + int(row.selected_index))
    choices = [i for i in range(36) if i != int(row.top_patch)]
    return int(rng.choice(choices))

examples["random_patch"] = examples.apply(random_patch, axis=1)
examples.to_csv(OUTPUT / "semantic_probe_examples.csv", index=False)

dataset = load_dataset("lmms-lab/POPE", split="test")

def get_image(row_idx):
    image = dataset[int(row_idx)]["image"]
    return image.convert("RGB")

print(examples.groupby(["model_name","split","clean_group"]).size())


In [ ]:
# Cell 3: shared Qwen semantic-probe implementation

from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

COCO_OBJECTS = [
    "person","bicycle","car","motorcycle","airplane","bus","train","truck","boat",
    "traffic light","fire hydrant","stop sign","parking meter","bench","bird","cat",
    "dog","horse","sheep","cow","elephant","bear","zebra","giraffe","backpack",
    "umbrella","handbag","tie","suitcase","frisbee","skis","snowboard","sports ball",
    "kite","baseball bat","baseball glove","skateboard","surfboard","tennis racket",
    "bottle","wine glass","cup","fork","knife","spoon","bowl","banana","apple",
    "sandwich","orange","broccoli","carrot","hot dog","pizza","donut","cake","chair",
    "couch","potted plant","bed","dining table","toilet","tv","laptop","mouse",
    "remote","keyboard","cell phone","microwave","oven","toaster","sink",
    "refrigerator","book","clock","vase","scissors","teddy bear","hair drier",
    "toothbrush","vehicle","table","phone","sofa","purse","bag","plane","aircraft"
]

def nested_getattr(obj, path):
    cur = obj
    for part in path.split("."):
        if not hasattr(cur, part):
            return None
        cur = getattr(cur, part)
    return cur

def locate_final_norm(model):
    paths = [
        "model.norm",
        "model.language_model.norm",
        "model.language_model.model.norm",
        "language_model.norm",
        "language_model.model.norm",
    ]
    for path in paths:
        module = nested_getattr(model, path)
        if isinstance(module, torch.nn.Module):
            return path, module
    candidates = []
    hidden_size = model.get_output_embeddings().weight.shape[1]
    for name, module in model.named_modules():
        weight = getattr(module, "weight", None)
        if (
            "norm" in name.lower()
            and isinstance(weight, torch.Tensor)
            and weight.ndim == 1
            and weight.numel() == hidden_size
        ):
            candidates.append((name,module))
    if not candidates:
        return "identity", torch.nn.Identity()
    candidates.sort(key=lambda x: -x[0].count("."))
    return candidates[0]

def prompt_text(question):
    return "Answer only yes or no based only on the visible image.\nQuestion: " + str(question)

def prepare_inputs(processor, image, question, device):
    messages = [{
        "role":"user",
        "content":[
            {"type":"image","image":image},
            {"type":"text","text":prompt_text(question)},
        ],
    }]
    try:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
    )
    return {k:(v.to(device) if hasattr(v,"to") else v) for k,v in inputs.items()}

def get_image_token_positions(model, processor, input_ids):
    candidates = []
    for obj in [model.config, getattr(model.config,"text_config",None)]:
        if obj is None:
            continue
        for attr in ["image_token_id","vision_token_id"]:
            value = getattr(obj, attr, None)
            if isinstance(value, int):
                candidates.append(value)
    token = getattr(processor, "image_token", None)
    if token:
        token_id = processor.tokenizer.convert_tokens_to_ids(token)
        if isinstance(token_id, int):
            candidates.append(token_id)
    for token_id in sorted(set(candidates)):
        positions = torch.nonzero(input_ids[0] == token_id, as_tuple=False).flatten()
        if len(positions) >= 4:
            return positions
    # Fallback to token strings containing image_pad.
    ids = input_ids[0].detach().cpu().tolist()
    toks = processor.tokenizer.convert_ids_to_tokens(ids)
    positions = [i for i,t in enumerate(toks) if "image" in str(t).lower()]
    if len(positions) >= 4:
        return torch.tensor(positions, device=input_ids.device)
    raise RuntimeError("Could not identify Qwen visual-token positions.")

def infer_visual_grid(inputs, n_visual_tokens, model):
    grid = inputs.get("image_grid_thw")
    merge = getattr(getattr(model.config,"vision_config",None), "spatial_merge_size", 2)
    if grid is not None:
        t,h,w = [int(x) for x in grid[0].detach().cpu().tolist()]
        gh, gw = max(1,h//merge), max(1,w//merge)
        if gh*gw == n_visual_tokens:
            return gh,gw
    # factorization fallback
    pairs=[]
    for h in range(1,int(math.sqrt(n_visual_tokens))+1):
        if n_visual_tokens%h==0:
            pairs.extend([(h,n_visual_tokens//h),(n_visual_tokens//h,h)])
    return min(pairs,key=lambda p:abs(p[0]-p[1])) if pairs else (1,n_visual_tokens)

def token_indices_for_patch(patch, gh, gw):
    x,y = int(patch)%6, int(patch)//6
    x1,x2 = math.floor(x*gw/6), math.ceil((x+1)*gw/6)
    y1,y2 = math.floor(y*gh/6), math.ceil((y+1)*gh/6)
    return [yy*gw+xx for yy in range(y1,min(y2,gh)) for xx in range(x1,min(x2,gw))]

def concept_token_map(tokenizer, concepts):
    mapping={}
    for concept in concepts:
        ids=set()
        variants={concept,concept.lower(),concept.capitalize()," "+concept," "+concept.lower()}
        for v in variants:
            enc=tokenizer.encode(v,add_special_tokens=False)
            if len(enc) <= 3:
                ids.update(int(x) for x in enc)
        if ids:
            mapping[concept]=sorted(ids)
    return mapping

@torch.inference_mode()
def run_model_probe(model_id, frame, output_dir):
    print("Loading", model_id)
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    model.eval()
    device = next(model.parameters()).device
    norm_name, final_norm = locate_final_norm(model)
    final_norm = final_norm.to(device)
    lm_head = model.get_output_embeddings()
    concept_map = concept_token_map(processor.tokenizer, COCO_OBJECTS)
    probe_ids = sorted(set(i for ids in concept_map.values() for i in ids))
    id_to_col = {token_id:i for i,token_id in enumerate(probe_ids)}
    weights = lm_head.weight[
        torch.tensor(probe_ids,device=lm_head.weight.device)
    ].detach().float()

    result_path = output_dir / f"{frame.model_name.iloc[0]}_semantic_probe.csv"
    done = pd.read_csv(result_path) if result_path.exists() else pd.DataFrame()
    completed = set(done.get("selected_index",pd.Series(dtype=int)).astype(int).tolist())
    rows=[]

    for _,meta in tqdm(
        frame[~frame.selected_index.astype(int).isin(completed)].iterrows(),
        total=(~frame.selected_index.astype(int).isin(completed)).sum(),
        desc=str(frame.model_name.iloc[0]),
    ):
        image=get_image(int(meta.row_idx))
        inputs=prepare_inputs(processor,image,meta.question,device)
        outputs=model(**inputs,output_hidden_states=True,use_cache=False,return_dict=True)
        positions=get_image_token_positions(model,processor,inputs["input_ids"])
        nvis=len(positions)
        gh,gw=infer_visual_grid(inputs,nvis,model)
        top_local=token_indices_for_patch(int(meta.top_patch),gh,gw)
        rand_local=token_indices_for_patch(int(meta.random_patch),gh,gw)
        hidden_states=outputs.hidden_states
        n_layers=len(hidden_states)

        target=str(meta.target_category)
        target_ids=concept_map.get(target,[])
        target_cols=[id_to_col[i] for i in target_ids if i in id_to_col]

        for layer,hs in enumerate(hidden_states):
            visual=hs[0,positions,:].float()
            valid_top=[i for i in top_local if i<len(visual)]
            valid_rand=[i for i in rand_local if i<len(visual)]
            if not valid_top or not valid_rand:
                continue
            vt=visual[valid_top].mean(0,keepdim=True)
            vr=visual[valid_rand].mean(0,keepdim=True)
            lt=(final_norm(vt) @ weights.T)[0].detach().cpu().numpy()
            lr=(final_norm(vr) @ weights.T)[0].detach().cpu().numpy()
            target_top=float(np.max(lt[target_cols])) if target_cols else np.nan
            target_rand=float(np.max(lr[target_cols])) if target_cols else np.nan

            concept_scores=[]
            for concept,ids in concept_map.items():
                cols=[id_to_col[i] for i in ids if i in id_to_col]
                if cols:
                    concept_scores.append((concept,float(np.max(lt[cols]))))
            concept_scores.sort(key=lambda x:x[1],reverse=True)

            rows.append({
                **meta.to_dict(),
                "layer":layer,
                "layer_fraction":layer/max(1,n_layers-1),
                "n_visual_tokens":nvis,
                "visual_grid_h":gh,
                "visual_grid_w":gw,
                "target_logit_top":target_top,
                "target_logit_random":target_rand,
                "target_margin":target_top-target_rand,
                "top_concepts_json":json.dumps(concept_scores[:10]),
                "final_norm_path":norm_name,
            })

        done=pd.concat([done,pd.DataFrame(rows)],ignore_index=True)
        done=done.drop_duplicates(["selected_index","layer"],keep="last")
        done.to_csv(result_path,index=False)
        rows=[]
        del outputs,inputs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    del model,processor,weights,lm_head,final_norm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.read_csv(result_path)

print("Implementation ready.")


In [ ]:
# Cell 4: run Qwen 7B, select semantic layer on 7B validation only

data7 = examples[examples.model_name=="qwen7b"].copy()
probe7 = run_model_probe(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    data7,
    OUTPUT,
)

val7 = probe7[probe7.split.astype(str)=="val"].copy()
if val7.empty:
    # Earlier 7B package primarily used eval labels; use half the matched pairs
    # deterministically as development without looking at outcomes.
    ids=sorted(probe7.selected_index.unique().tolist())
    dev=set(ids[::2])
    val7=probe7[probe7.selected_index.isin(dev)].copy()
    probe7["semantic_split"]=np.where(probe7.selected_index.isin(dev),"dev","eval")
else:
    probe7["semantic_split"]=probe7.split

layer7 = (
    val7.groupby("layer")
    .agg(
        mean_target_margin=("target_margin","mean"),
        positive_rate=("target_margin",lambda x:float((np.asarray(x)>0).mean())),
        n=("selected_index","nunique"),
    )
    .reset_index()
    .sort_values(["mean_target_margin","positive_rate"],ascending=[False,False])
)
SELECTED_LAYER_7B=int(layer7.iloc[0].layer)
N_LAYERS_7B=int(probe7.layer.max()+1)
SELECTED_FRACTION=SELECTED_LAYER_7B/max(1,N_LAYERS_7B-1)

layer7["selected"]=(layer7.layer.astype(int)==SELECTED_LAYER_7B).astype(int)
layer7.to_csv(OUTPUT/"qwen7b_semantic_layer_selection.csv",index=False)
probe7.to_csv(OUTPUT/"qwen7b_semantic_probe.csv",index=False)

print("Selected 7B semantic layer:",SELECTED_LAYER_7B)
print("Selected relative depth:",SELECTED_FRACTION)
print(layer7.head(10).to_string(index=False))


In [ ]:
# Cell 5: run Qwen 3B and freeze the 7B relative-depth rule

data3 = examples[examples.model_name=="qwen3b"].copy()
probe3 = run_model_probe(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    data3,
    OUTPUT,
)
N_LAYERS_3B=int(probe3.layer.max()+1)
TRANSFER_LAYER_3B=int(round(SELECTED_FRACTION*max(1,N_LAYERS_3B-1)))

selected7=probe7[probe7.layer.astype(int)==SELECTED_LAYER_7B].copy()
selected3=probe3[probe3.layer.astype(int)==TRANSFER_LAYER_3B].copy()
selected7["probe_setting"]="7b_dev_selected"
selected3["probe_setting"]="frozen_relative_depth_transfer"

selected_all=pd.concat([selected7,selected3],ignore_index=True)
selected_all.to_csv(OUTPUT/"selected_semantic_probe_results.csv",index=False)

print("7B layer:",SELECTED_LAYER_7B)
print("3B frozen transfer layer:",TRANSFER_LAYER_3B)


In [ ]:
# Cell 6: evaluate, generate diagnosis tables and figures, export

from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

rows=[]
for model_name,frame in selected_all.groupby("model_name"):
    eval_frame=frame[
        frame.get("semantic_split",frame.split).astype(str).isin(["eval","test"])
    ].copy()
    if eval_frame.empty:
        eval_frame=frame.copy()

    hall=eval_frame[eval_frame.clean_group.astype(str)=="hallucinated_yes"]
    ground=eval_frame[eval_frame.clean_group.astype(str)=="grounded_yes"]
    y=eval_frame.is_grounded.astype(int).to_numpy()
    score=eval_frame.target_margin.astype(float).to_numpy()
    valid=np.isfinite(score)

    result={
        "model_name":model_name,
        "n":int(len(eval_frame)),
        "grounded_n":int(len(ground)),
        "hallucinated_n":int(len(hall)),
        "mean_target_margin_grounded":float(ground.target_margin.mean()),
        "mean_target_margin_hallucinated":float(hall.target_margin.mean()),
        "top_patch_beats_random_rate":float((eval_frame.target_margin>0).mean()),
    }
    if valid.sum()>1 and len(np.unique(y[valid]))==2:
        result["grounded_vs_hallucinated_auroc"]=float(roc_auc_score(y[valid],score[valid]))
        result["grounded_vs_hallucinated_auprc"]=float(average_precision_score(y[valid],score[valid]))
    try:
        result["top_vs_random_wilcoxon_p"]=float(
            wilcoxon(
                eval_frame.target_logit_top.astype(float),
                eval_frame.target_logit_random.astype(float),
                alternative="greater",
            ).pvalue
        )
    except Exception:
        result["top_vs_random_wilcoxon_p"]=None
    rows.append(result)

summary=pd.DataFrame(rows)
summary.to_csv(OUTPUT/"semantic_probe_summary.csv",index=False)

def diagnosis(row):
    concepts=[]
    try:
        concepts=[x[0] for x in json.loads(row.top_concepts_json)]
    except Exception:
        pass
    target=str(row.target_category).lower()
    target_seen=any(target in str(c).lower() or str(c).lower() in target for c in concepts[:5])
    if str(row.clean_group)=="grounded_yes":
        return "grounded_control"
    if float(row.target_margin)>0 and target_seen:
        return "localized_semantic_false_cue"
    if float(row.target_margin)>0:
        return "causal_patch_contains_target_like_signal"
    return "diffuse_or_unresolved"

selected_all["automatic_diagnosis"]=selected_all.apply(diagnosis,axis=1)
selected_all.to_csv(OUTPUT/"qwen_semantic_hallucination_diagnoses.csv",index=False)

fig,ax=plt.subplots(figsize=(8,5))
plot=summary.set_index("model_name")
ax.bar(plot.index,plot["top_patch_beats_random_rate"])
ax.set_ylim(0,1)
ax.set_ylabel("Causal patch target-logit > random patch")
ax.set_title("Semantic readout of Qwen causal visual patches")
fig.tight_layout()
fig.savefig(OUTPUT/"semantic_probe_top_vs_random.png",dpi=220,bbox_inches="tight")
plt.close(fig)

final={
    "selected_7b_layer":int(SELECTED_LAYER_7B),
    "selected_relative_depth":float(SELECTED_FRACTION),
    "frozen_3b_transfer_layer":int(TRANSFER_LAYER_3B),
    "results":summary.to_dict(orient="records"),
    "interpretation_note":"Post-merger visual-token logit-lens readout. Treat as a semantic probe, not literal chain of thought."
}
with open(OUTPUT/"final_semantic_probe_result.json","w") as f:
    json.dump(final,f,indent=2)

package=PROJECT/"qwen_semantic_causal_hallucination_diagnosis_RESULTS"
if package.exists():
    shutil.rmtree(package)
package.mkdir(parents=True)
for p in OUTPUT.iterdir():
    if p.is_file():
        shutil.copy2(p,package/p.name)
archive=shutil.make_archive(
    str(WORKSPACE/"qwen_semantic_causal_hallucination_diagnosis_RESULTS"),
    "zip",
    root_dir=package,
)
print(summary.to_string(index=False))
print("\nOutput:",archive)
